# rank0-only-side-effects — ex1: guard checkpoint + log side effects behind if rank == 0

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rank0-only-side-effects`. Running the final beacon cell reports progress against the `Distributed: rank-0-only side effects` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Distributed: rank-0-only side effects — quick refresher

Anything that touches a SHARED RESOURCE (filesystem, network, wandb, tqdm, print to stdout) must run on exactly one rank — conventionally rank 0:

```python
if rank == 0:
    torch.save(model.state_dict(), 'ckpt.pt')
    wandb.log({'loss': loss.item()})
    print(f'epoch {epoch}: loss={loss.item():.4f}')
```

**Why.** Without the guard, every rank writes to `ckpt.pt` — N concurrent writes race for the same path, the file ends up corrupted, and you've also wasted N times the wandb quota. Same for stdout: log lines interleave from N processes and become unreadable.

**Things that do NOT need the guard.** Anything that's process-local: per-rank tensors, per-rank `.grad`, the model forward pass, the optimizer step. Every rank computes its own grads on its own shard.

**Things that DO need the guard.**
- File writes (`torch.save`, `open('w')`, `csv.writer`).
- Network calls (wandb, MLflow, http POST).
- `print` / `logging` / `tqdm` (or use a logger that filters by rank).
- Mutating shared filesystem state (creating directories, downloading datasets — race on first call).

**Pair with `dist.barrier()` when other ranks depend on the work.** If rank 0 downloads a dataset that rank 1 will read, rank 1 must wait until rank 0 finishes — otherwise rank 1 tries to read a half-written file. The idiom: rank 0 does the work, then `barrier`; rank > 0 hits `barrier` first, blocks, then proceeds.

### Exercise 1 — guard checkpoint + log side effects behind if rank == 0

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `if rank == 0:` guard around shared-resource writes (checkpoint, log) so the action runs exactly once across the world, while per-rank compute proceeds on every rank.
> Keywords: rank-0, side-effects, checkpoint, logging
> ```

**KCs targeted:** `rank-0-guard-side-effects`, `shared-resource-singleton-writer`

Implement `ex1_epoch_end(rank, world_size, model_state, loss, ckpt_writer, log_writer, per_rank_recorder)`. The canonical end-of-epoch hook used in every distributed training script.

Behavior — ON EVERY RANK:
  - Call `per_rank_recorder(rank, loss)` to record that this rank did its forward/backward (every rank's contribution matters).

Behavior — ON RANK 0 ONLY:
  - Call `ckpt_writer(model_state)`.
  - Call `log_writer(f'loss={loss:.4f}')`.

Wrap the rank-0 calls in `if rank == 0:`. Do NOT use `if rank % world_size == 0:` or any other workaround — the convention is exactly `rank == 0`.

Inputs:
- `rank`, `world_size`: ints.
- `model_state`: arbitrary object (the checkpoint).
- `loss`: float.
- `ckpt_writer`, `log_writer`, `per_rank_recorder`: callbacks.

Output: `None`.

The test calls `ex1_epoch_end` for each `rank` in `range(world_size)` (simulated locally — no real multiprocessing) using mock callbacks, and verifies the guard fired correctly.

In [ ]:
def ex1_epoch_end(rank, world_size, model_state, loss,
                  ckpt_writer, log_writer, per_rank_recorder):
    # Every rank records its loss (per-rank, no race).
    per_rank_recorder(rank, loss)

    # Shared-resource side effects: rank 0 only.
    if rank == 0:
        ckpt_writer(model_state)
        log_writer(f'loss={loss:.4f}')


<details><summary>Solution</summary>

```python
def ex1_epoch_end(rank, world_size, model_state, loss,
                  ckpt_writer, log_writer, per_rank_recorder):
    # Every rank records its loss (per-rank, no race).
    per_rank_recorder(rank, loss)

    # Shared-resource side effects: rank 0 only.
    if rank == 0:
        ckpt_writer(model_state)
        log_writer(f'loss={loss:.4f}')
```

**The `if rank == 0` idiom is universal.** PyTorch's own distributed examples, fairseq, ARENA's `DistResNetTrainer`, every reference implementation — all gate checkpoint/log behind the same one-line guard. Reviewers expect it.

**Why not `if dist.get_rank() == 0`?** Same effect, but now your function only works AFTER `init_process_group` has been called. Taking `rank` as a parameter is more testable (no global state) and works with single-process world_size=1 paths too.

**What about `if rank == 0 and step % N == 0:` for throttled logging.** Compose the two conditions. The rank-0 guard is orthogonal to the throttle — apply both.

**Pair with `dist.barrier()` when ordering matters.** If rank 0 writes a checkpoint that another rank will load (rare in training, common in eval), add `dist.barrier()` after the write so the loaders don't race the writer.

**Don't put the COMPUTE inside the rank-0 guard.** The FORWARD pass, the loss computation, the backward pass — every rank does all of these. Only the SIDE EFFECTS (filesystem, network) need the guard.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()